MatterGen

In [ ]:
https://github.com/microsoft/mattergen/tree/main/benchmark

Chemeleon

In [ ]:
from chemeleon_dng.sample import sample

sample(
    task="dng",
    num_samples=2000,
    batch_size=40,
    output_dir="/Users/zhenzhu/Documents/manifish_project/generated_strucs/chemeleon_strucs/",
    device="cpu"
)

In [ ]:
#read the json file and get the cifs generated
import json
path = '/Users/zhenzhu/Documents/manifish_project/generated_strucs/chemeleon_strucs/chemeleon_dng_mp_20_v0.0.2.json'
with open(path, 'r') as f:
    data = json.load(f)

{'@module': 'pymatgen.core.structure', '@class': 'Structure', 'charge': 0.0, 'lattice': {'matrix': [[3.4967423346097197, 0.0, -1.1075747272589063], [-0.43106821809293094, 3.702134133415334, -1.3202402996639084], [0.0, 0.0, 6.080318691108359]], 'pbc': [True, True, True], 'a': 3.667959750476343, 'b': 3.954067702923834, 'c': 6.080318691108359, 'alpha': 109.50530926584294, 'beta': 107.57532405775656, 'gamma': 90.17804890697616, 'volume': 78.71221323531081}, 'properties': {}, 'sites': [{'species': [{'element': 'Li', 'occu': 1.0}], 'abc': [0.39840593934059143, 0.3632477223873139, 0.0516681931912899], 'properties': {}, 'label': 'Li1', 'xyz': [1.2365383660363831, 1.3447917919354522, -0.6066795506635458]}, {'species': [{'element': 'Th', 'occu': 1.0}], 'abc': [0.7341293692588806, 0.6979328393936157, 0.7207009792327881], 'properties': {}, 'label': 'Th1', 'xyz': [2.2662045791419136, 2.583840987550587, 2.6475494377732516]}, {'species': [{'element': 'Ge', 'occu': 1.0}], 'abc': [0.06770162284374237, 

In [ ]:
#use ase to get atoms from the lattice and sites info of each data point in data
from ase import Atoms
from ase.io import write
import numpy as np
from mace.calculators import MACECalculator, mace_mp
import torch

calculator = mace_mp(model='/Users/zli6/Documents/adit/mace/models/mp0a/model_advance_cpu.model', device='cpu')
for i, entry in enumerate(data):
    lattice = entry['lattice']
    print("Lattice:", lattice)
    sites = entry['sites'] #list of dicts
    symbols = [site['species'][0]['element'] for site in sites]
    positions = [site['abc'] for site in sites]
    cell = lattice['matrix']
    atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=True)




Lattice: {'matrix': [[3.4967423346097197, 0.0, -1.1075747272589063], [-0.43106821809293094, 3.702134133415334, -1.3202402996639084], [0.0, 0.0, 6.080318691108359]], 'pbc': [True, True, True], 'a': 3.667959750476343, 'b': 3.954067702923834, 'c': 6.080318691108359, 'alpha': 109.50530926584294, 'beta': 107.57532405775656, 'gamma': 90.17804890697616, 'volume': 78.71221323531081}
Atoms(symbols='LiThGe2N', pbc=True, cell=[[3.4967423346097197, 0.0, -1.1075747272589063], [-0.43106821809293094, 3.702134133415334, -1.3202402996639084], [0.0, 0.0, 6.080318691108359]])


In [ ]:
import pandas as pd
from ase.io import read
import numpy as np
from mace.calculators import MACECalculator, mace_mp
import torch

#df_train = pd.read_csv('/Users/zli6/Documents/adit/mace/models/data/mp20/train.csv')
train = read("/Users/zli6/Documents/adit/mace/models/mp0a/train.xyz", index=":")

# print(len(train))

calculator = mace_mp(model='/Users/zli6/Documents/adit/mace/models/mp0a/model_advance_cpu.model', device='cpu')
#calculator = mace_mp(model='/Users/zli6/Documents/adit/mace/models/mp0a/medium_2023-12-03-mace-128-L1_epoch-199.model', device='cpu')
#calculator = MACECalculator(model_paths='/Users/zli6/Documents/adit/mace/models/mp0a/large_2024-01-07-mace-128-L2_epoch-199.model', device='cpu')
# all_descriptors_medium = []
# all_descriptors_large = []
all_descriptors_mpa = []
all_atoms_number = []
#convert ase atoms to cif
for i, atoms in enumerate(train):
    atoms.write("cif_file.cif")
#for cif in df_train["cif"]:
#for cif in valid_sample_atoms[4:]:
    # with open('cif_file', 'w') as f:
    #     f.write(cif)
    init_conf=read('cif_file.cif', format='cif')
    #init_conf = atoms
    all_atoms_number.append(init_conf.get_atomic_numbers())
    # descriptors = calculator.get_descriptors(init_conf, num_layers=1)
    # all_descriptors_medium.append(descriptors)
    descriptors = calculator.get_descriptors(init_conf, num_layers=1)
    all_descriptors_mpa.append(descriptors)
#f.close()
import pickle
# with open('medium_embeddings.pkl','wb') as f:
#     pickle.dump(all_descriptors_medium, f)
with open('mh_atomnumbers_all_CuAuCu.pkl','wb') as f:
    pickle.dump(all_atoms_number, f)
with open('mh_embeddings_all_CuAuCu.pkl','wb') as f:
    pickle.dump(all_descriptors_mpa, f)